📝
# FieldCare — Sprint 1 Live notebook

Use this **one Live notebook for all four live sessions in this sprint**. Your Campus lessons have their own Campus notebook. Save a personal Drive copy of each and return to the same copies throughout the sprint.

HelioDesk’s FieldCare service is our shared case. The workshops let you apply the taught patterns, inspect actual observations and defend your decisions with a peer. Each session below names its starting state, task, evidence and debrief.

## Start and preserve your work

1. Select a Python CPU runtime and run setup and imports once per runtime.
2. Upload your current Campus source/evidence checkpoint in Colab’s Files panel. Set `CAMPUS_CHECKPOINT` to its path. A checkpoint is your saved service files, not the notebook itself.
3. Run the workspace cell. It restores into an isolated Live workspace, preserving the Campus copy. With no checkpoint, `USE_DEMO_RECOVERY=True` explicitly selects a labelled public practice state; it does not establish completion of Campus work.
4. Use the table of contents to open the current session. Run its preparation cells before its task. Later sessions build on earlier live sections unless a supplied checkpoint already provides that starting state.
5. Write your own predictions and explanations. Blank decisions stay pending; **Run all does not complete your work**. Public practice never completes the reserved Campus assessment.

Use a new `RUN_NAME` when importing a newer checkpoint; existing work is deliberately preserved. `RESUME_ZIP` reopens an earlier Live export. Never paste keys into cells or outputs. These workshops use demo/transport fixtures by default, not real model output.


## Instructor guides and live decks in Notion

- [LS 1 — Application vs service and REST orientation](https://app.notion.com/p/3ea5a55972ad8125af2edd3fa083a7d9?pvs=204)
- [LS 2 — Extending the scaffold](https://app.notion.com/p/3ea5a55972ad817d8151ff4a5cb6bdcc?pvs=204)
- [LS 3 — Input/output contracts and versioning](https://app.notion.com/p/3ea5a55972ad8159b1ffd860117002d4?pvs=204)
- [LS 4 — Sprint 1 checkpoint](https://app.notion.com/p/3ea5a55972ad81da914cc512dec06b48?pvs=204)


In [ ]:
#@title Load the shared course code from GitHub
from pathlib import Path
import os, re, sys, subprocess, tempfile

# Reviewed shared helpers: this immutable revision keeps every lesson on the same source.
GITHUB_REPO = 'https://github.com/richhiey/ai-app-dev_Mod-B.git'
GITHUB_REF = 'fb9d95b1cadcc29972cb9dd7388201aaa5868e1d'
repo_url = os.environ.get('MODULE_B_REPO_URL', GITHUB_REPO).strip()
repo_ref = os.environ.get('MODULE_B_REPO_REF', GITHUB_REF).strip()
destination = Path(os.environ.get('MODULE_B_REPO', '/content/ai-app-dev_Mod-B')).expanduser()
candidates = [destination, Path.cwd(), *Path.cwd().resolve().parents]
REPO = next((p.resolve() for p in candidates if (p/'src/module_b/notebook.py').is_file()), None)

if REPO is None:
    if not re.fullmatch(r'https://github\.com/[A-Za-z0-9_.-]+/[A-Za-z0-9_.-]+(?:\.git)?', repo_url):
        raise RuntimeError('The course author must set GITHUB_REPO to the published HTTPS GitHub repository URL.')
    if not re.fullmatch(r'[0-9a-fA-F]{40}|refs/tags/[A-Za-z0-9][A-Za-z0-9._/-]*', repo_ref):
        raise RuntimeError('The course author must set GITHUB_REF to a release tag (refs/tags/...) or full commit SHA.')
    if destination.exists():
        raise FileExistsError('The destination already exists without the expected helpers. Preserve it and use a fresh runtime or a new MODULE_B_REPO path.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Prepare separately, so a failed download cannot replace existing student work.
    with tempfile.TemporaryDirectory(prefix='module-b-download-', dir=destination.parent) as temporary:
        checkout = Path(temporary)/'checkout'
        subprocess.run(['git','clone','--no-checkout','--depth','1',repo_url,str(checkout)],check=True)
        subprocess.run(['git','-C',str(checkout),'fetch','--depth','1','origin',repo_ref],check=True)
        subprocess.run(['git','-C',str(checkout),'checkout','--detach','FETCH_HEAD'],check=True)
        if not (checkout/'src/module_b/notebook.py').is_file() or not (checkout/'requirements.lock').is_file():
            raise RuntimeError('The selected repository revision does not contain the expected Module B source layout.')
        checkout.rename(destination)
    REPO = destination.resolve()

# Existing local checkouts are reused; setup never pulls, resets or overwrites them.
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'-c',str(REPO/'requirements.lock')],check=True)
if str(REPO/'src') not in sys.path:
    sys.path.insert(0,str(REPO/'src'))
from module_b import __version__
print('Shared package ready:', __version__)
print('Source checkout:', REPO)

In [ ]:
import json
import httpx
from module_b.workspace import prepare_example, restore_workspace
from module_b.data import load_fixture, load_json
from module_b.source import source_excerpt
from module_b.edits import write_source
from module_b.fieldcare import demo_service, trace_paths, verify_prompt_binding, inspect_route_bindings
from module_b.contracts import inspect_contract
from module_b.versioning import inspect_version_bindings
from module_b.notebook import (table, apply_student_edits, register_source, baseline,
    observe_change, revision, fresh, provider_service, export_progress, preserved)

In [ ]:
RUN_NAME = 'sprint-1-live'
CAMPUS_CHECKPOINT = ''
USE_DEMO_RECOVERY = True
RESUME_ZIP = ''  # Example: /content/sprint-1-checkpoint-....zip
work = REPO/'work'/RUN_NAME
project = work/'project'
incoming = RESUME_ZIP or CAMPUS_CHECKPOINT
if not incoming and not USE_DEMO_RECOVERY:
    raise ValueError('Set a Campus checkpoint, or explicitly select demo recovery.')
if incoming and not project.exists():
    project = restore_workspace(incoming, project)
print('Origin:', 'imported checkpoint; existing Live workspace preserved on rerun' if incoming else 'public demo recovery; Campus work remains pending')
project = prepare_example(destination=project, repo_root=REPO)
lab = prepare_example(destination=work/'lab', repo_root=REPO)
accepted = load_fixture(project, 'valid-request')
incomplete = load_fixture(project, 'incomplete-request')
initial = baseline(project, 'sprint1-entry', ['/v1/diagnose'])
evidence = {}
print('Your project:', project)
print('Worked-example lab:', lab)

## Live workshops · one notebook, four sessions

Run setup, imports and the workspace cell above, then use these four workshop sections throughout the sprint. The Campus exercises remain in your separate Campus notebook; imported project work is kept in this isolated Live copy. This lane never completes the supervisor assessment for you.

FieldCare is the supplied synthetic service for HX filter/airflow questions. A software caller sends a question and optional equipment identifier; the service validates the request, gathers the limited evidence and either asks for context or reaches the model adapter. The demonstration adapter returns labelled fixed text and makes no external call.

| Session | Live purpose | Individual evidence |
|---|---|---|
| LS01 | Trace and diagnose a request | Four code locations, stop boundary and caller consequence |
| LS02 | Test your extension as someone else's client | Contract card, selected prompt and peer-client report |
| LS03 | Defend old/new compatibility | Two agreements, independent bindings and migration note |
| LS04 | Demonstrate and repair one claim | Public evidence index, peer challenge and repeated check |

The separate `live_lab` is public instructional material. Your existing `project` remains your work. All live records save into its final checkpoint. `ready_for_discussion` means observations and written reasoning exist; it is not a correctness judgment or grade. If you use a public lab or recorded trace, label it and keep your own execution pending.


In [ ]:
from module_b.live_sessions import save_record, read_records, record_is_current
live_lab = prepare_example(destination=work/'live-lab', repo_root=REPO)
live_request = {'question':'Which HX airflow and filter checks are documented for this unit?', 'equipment_id':'EQ-FC-1002'}
live_fresh_request = {'question':'What filter orientation and panel seating checks should I verify?', 'equipment_id':'EQ-FC-1002'}
live_incomplete = {'question':live_request['question']}
# Public fixtures for the separate in-process trace. These never replace project fixtures.
for name, payload in [('valid',live_request),('invalid',{}),('incomplete',live_incomplete)]:
    (live_lab/'fixtures'/f'{name}-request.json').write_text(json.dumps(payload,indent=2)+'\n')
print('Public lab:',live_lab)
print('Your retained service:',project)
print('Previously saved live records:', list(read_records(project)))


### LS01 · Predict, call, locate

**Action:** predict the method, route and response fields before executing the cell. **Reason:** the caller needs an agreement that another program can consume. An endpoint is an HTTP method plus a service path; a request body carries the caller's data.

**Expected:** the supported request returns `200`, `status='ready'`, and the four fields `answer`, `status`, `citations`, `mode`. This verifies actual local HTTP in demo mode. The health response only establishes local readiness, not provider availability.

At minute 12, show the method/path/body annotation. At minute 21, point to the response declaration that supports one caller expectation. A browser's address bar does not send this JSON POST.


In [ ]:
ls01_http = []
with demo_service(live_lab) as service:
    assert httpx.get(service.base_url+'/health').status_code == 200
    for label, payload in [('supported',live_request),('malformed',{}),('clarification',live_incomplete)]:
        response = httpx.post(service.base_url+'/v1/diagnose',json=payload)
        row={'case':label,'status':response.status_code,'body':response.json()}
        ls01_http.append(row)
        print(label, response.status_code, response.json())
for file, symbol in [('app/routes.py','diagnose'),('app/schemas.py',None),('app/service.py','run_diagnosis'),('app/model.py','generate_answer')]:
    print(file); print(source_excerpt(live_lab,file,symbol))


### LS01 · Validation and clarification stop at different boundaries

The malformed request deliberately omits the required question. A `422` response reports invalid input before the handler. Omitting optional equipment context is valid input; application logic returns `needs_clarification` without reaching the adapter.

**Action:** inspect the separate trace below. **Reason:** HTTP status alone does not reveal which function ran. **Expected:** supported / malformed / clarification requests reach the adapter `1 / 0 / 0` times. External provider calls remain zero. These are newly executed in-process requests, not counters attached to the earlier TCP requests.


In [ ]:
ls01_trace = trace_paths(live_lab)
table(ls01_trace,['case','status_code','handler_service_calls','adapter_calls','provider_calls'])


### LS01 · Your trace, 14 minutes

**Goal:** explain what a caller can rely on for a fresh supported request. **Start:** `live_fresh_request` and the unchanged public lab. **Constraints:** keep source unchanged; distinguish HTTP observation, code inspection and the separate trace; do not claim a provider call. **Deliver:** a route/input/output/adapter map, the observed response state and one consequence for the caller. Choose and execute your own confirming request below.

Use the learner exercise's blank trace card. Your instructor checks one boundary halfway through. After the attempt, compare the public reference supplied with the session. Save your reasoning in the fields below; blank fields remain pending.


In [ ]:
ls01_fresh_observation = {}
# Set True after making your prediction; the cell is a fresh actual HTTP observation.
RUN_LS01_TASK = False
if RUN_LS01_TASK:
    with demo_service(live_lab) as service:
        response = httpx.post(service.base_url+'/v1/diagnose',json=live_fresh_request)
        ls01_fresh_observation = {'status':response.status_code,'body':response.json()}
        print(ls01_fresh_observation)
ls01_reasoning = {'route_input_output_adapter':'', 'stop_boundary':'', 'caller_consequence':''}
ls01_record = save_record(project,1,observations=ls01_fresh_observation,reasoning=ls01_reasoning,
                         origin='public_practice',observed_project=live_lab)
print(ls01_record['review_status'])


### LS02 · The extension seam

A **prompt binding** is the particular instruction selected by a route when it calls the shared service. Read the supplied analogous route and register it in the public lab. It changes the requested answer format but preserves the input/output declarations. It is a worked example, not your dispatch implementation.

**Expected:** original and practice routes are callable. The isolated mocked-provider probe observes distinct prompt bindings. Demo answer wording stays fixed, so changed prose is not the test. Your own versioned dispatch route remains in `project` from prior work; if it is missing, use the earlier provided pattern to complete the bounded configuration rather than writing a framework.


In [ ]:
live_pattern = (REPO/'examples/patterns/practice_routes.py').read_text()
print(live_pattern)
write_source(live_lab,'app/practice_routes.py',live_pattern)
live_registration = '\nfrom app.practice_routes import router as practice_router\napp.include_router(practice_router)\n'
register_source(live_lab,live_registration)
ls02_demo = inspect_route_bindings(live_lab,{'/v1/diagnose':'app.routes','/practice/diagnose-brief':'app.practice_routes'})
assert ls02_demo['passed'], ls02_demo.get('errors')
print('Public pattern binding:',ls02_demo['passed'],'external calls:',ls02_demo['external_provider_calls'])


### LS02 · Peer-client contract review, 14 minutes

**Goal:** deliver your dispatch extension with a contract another caller can use. **Start:** your existing `/v1/dispatch-handover`, its route-local prompt and the `DiagnosticRequest`/`DiagnosticResponse` declarations. **Constraints:** retain `/v1/diagnose`, optional bounded equipment context, and valid clarification behavior. Do not hardcode a generated response. **Deliver:** one contract card and a client report containing accepted, malformed, missing-context and original-route observations, plus safe binding evidence.

Each partner owns their implementation. Exchange the contract card first, then let the client choose a public request that challenges it. Swap roles. Repair a mismatch if one exists; a justified no-change result is valid. Use `ls02_edits` only for your intended source changes. Never paste runtime secrets. Complete any generated-wording comparison in the existing provider section when access is available; label demo evidence precisely.


In [ ]:
ls02_edits = {}  # Optional deliberate source changes, keyed by relative app/*.py path.
ls02_registration = ''  # Only when your router is not already registered.
apply_student_edits(project,ls02_edits)
if ls02_registration.strip(): register_source(project,ls02_registration)
ls02_observations = {}
if (project/'app/handover_routes.py').is_file():
    ls02_bindings = inspect_route_bindings(project,{'/v1/diagnose':'app.routes','/v1/dispatch-handover':'app.handover_routes'})
    ls02_rows=[]
    with demo_service(project) as service:
        for path,payload,label in [('/v1/dispatch-handover',live_request,'accepted'),('/v1/dispatch-handover',{},'malformed'),('/v1/dispatch-handover',live_incomplete,'clarification'),('/v1/diagnose',live_request,'original')]:
            response=httpx.post(service.base_url+path,json=payload)
            ls02_rows.append({'case':label,'path':path,'status':response.status_code,'body':response.json()})
    table(ls02_rows,['case','path','status'])
    ls02_observations={'http':ls02_rows,'bindings':ls02_bindings,'preservation':preserved(project,initial)}
else:
    print('Own extension pending. Use the visible analogous pattern and retain your baseline.')
ls02_reasoning={'contract':'','peer_challenge':'','binding_evidence':'','repair_or_no_change':''}
print(save_record(project,2,observations=ls02_observations,reasoning=ls02_reasoning,origin='learner_service')['review_status'])


### LS03 · Compatibility before version labels

Compare these public change cards against your caller agreement:

- A spelling correction in a system instruction retains all promised fields and states. Check behavior; a new version is not automatically needed.
- A pilot accepts at most `600` question characters while the original accepts `2000`. An existing `900`-character caller would break if the tighter limit replaced its route.
- A prompt asks for a more confident answer but keeps the JSON shape. Structural validation cannot prove the answer remains grounded; behavior needs evaluation.

**Action:** register the analogous diagnostic pilot below and inspect independent bindings. **Reason:** changing only a path does not isolate mutable settings. **Expected:** the mocked probe changes the chosen new prompt/model binding while the old selection stays stable. The model labels inside this probe are synthetic wiring markers, not models to purchase or send to a provider.


In [ ]:
live_v2_source=(REPO/'examples/patterns/diagnose_v2_routes.py').read_text()
write_source(live_lab,'app/diagnose_v2_routes.py',live_v2_source)
register_source(live_lab,'\nfrom app.diagnose_v2_routes import router as diagnose_v2_router\napp.include_router(diagnose_v2_router)\n')
live_versions={'/v1/diagnose':{'module':'app.routes','model_env':'OPENROUTER_MODEL'},'/v2/diagnose':{'module':'app.diagnose_v2_routes','model_env':'FIELDCARE_DIAGNOSE_V2_MODEL'}}
ls03_demo=inspect_version_bindings(live_lab,versions=live_versions,changed_route='/v2/diagnose',accepted_payload=live_request)
assert ls03_demo['passed'],ls03_demo.get('errors')
for round_record in ls03_demo['rounds']: table(round_record['routes'])


### LS03 · Old-client review, 14 minutes

**Goal:** apply a pilot change while keeping the original caller usable. **Start:** your existing versioned handover and the `600`/`2000` input agreements. **Constraints:** change only the new version's intended binding, preserve response fields and valid clarification, and retain the old route. Do not create a third version merely to repeat the exercise. **Deliver:** before/after source or prompt fingerprints, accepted/rejected observations for both routes, and a migration note that names which caller has not moved.

Use the supplied version pattern if a binding is incomplete. Apply your own edits using the earlier versioning cells, then use the following checks. The synthetic client record is: pilot console uses `/v2`; offline tablet still uses `/v1`. Explain why a deprecation notice does not justify removal while that tablet still depends on the original contract.


In [ ]:
ls03_observations={}
if (project/'app/handover_v2_routes.py').is_file() and (project/'app/handover_routes.py').is_file():
    ls03_versions={'/v1/dispatch-handover':{'module':'app.handover_routes','model_env':'OPENROUTER_MODEL'},'/v2/dispatch-handover':{'module':'app.handover_v2_routes','model_env':'FIELDCARE_HANDOVER_V2_MODEL'}}
    ls03_bindings=inspect_version_bindings(project,versions=ls03_versions,changed_route='/v2/dispatch-handover',accepted_payload=live_request)
    ls03_rows=[]
    with demo_service(project) as service:
        for path in ls03_versions:
            for label,payload in [('supported',live_request),('old_900_character_caller',dict(live_request,question='filter '+'x'*893)),('missing_context',live_incomplete)]:
                response=httpx.post(service.base_url+path,json=payload)
                ls03_rows.append({'path':path,'case':label,'status':response.status_code,'state':response.json().get('status')})
    table(ls03_rows)
    ls03_observations={'http':ls03_rows,'bindings':ls03_bindings}
else:
    print('Own two-version evidence pending. Complete the supplied versioning pattern in your retained project.')
ls03_reasoning={'old_agreement':'','new_agreement':'','independence_evidence':'','migration_and_retirement':''}
print(save_record(project,3,observations=ls03_observations,reasoning=ls03_reasoning,origin='learner_service')['review_status'])


### LS04 · Demonstration and one bounded repair

Bring the familiar project's **public** exchanges and declarations, not reserved supervisor assessment payloads or solutions. Three eight-minute owner rounds give each person in a triad time to demonstrate, receive a challenge and record a response. Pair/remainder instructions are in the session guide.

Your caller reviewer asks whether the observed response matches its declared contract. Your evidence reviewer asks what proves the original caller still works. At each rotation, every owner records their own evidence. During the final **12-minute** individual block, repair one narrow mismatch or strengthen one missing check. A complete rebuild is outside the timebox.

Choose the public route you are demonstrating and predict its accepted, malformed and missing-context results. The next cell runs real local HTTP against that chosen path and the original. A route returning `404` is evidence to repair, not a reason to mark the review complete. Source/configuration evidence is still required alongside these statuses.


In [ ]:
ls04_route = ''  # Your existing versioned public-practice endpoint, not a reserved assessment route.
ls04_rows=[]
if ls04_route:
    with demo_service(project) as service:
        for path,payload,label in [(ls04_route,live_request,'accepted'),(ls04_route,{},'malformed'),(ls04_route,live_incomplete,'missing_context'),('/v1/diagnose',live_request,'original')]:
            response=httpx.post(service.base_url+path,json=payload)
            ls04_rows.append({'case':label,'path':path,'status':response.status_code,'body':response.json()})
    table(ls04_rows,['case','path','status'])
ls04_reasoning={'contract_and_binding_locations':'','peer_question':'','my_response':'','repair_or_no_change':'','repeated_observation':''}
print(save_record(project,4,observations=ls04_rows,reasoning=ls04_reasoning,origin='learner_service')['review_status'])
live_records=read_records(project)
print('Saved formative records:', {key:record['review_status'] for key,record in live_records.items()})
print('Save the notebook, then use the one final sprint export to preserve files and records.')


## Save your Live evidence and carry improvements back

Run this export after your session and again after later changes. It saves the Live workspace and the four public workshop records; it does not mark your Campus assessment complete. Download the printed ZIP from Colab’s Files panel and save this Live notebook separately.

**Apply a chosen improvement to Campus:** open your Campus copy, compare the changed source file with your own file, and apply only the change you can explain using the Campus editing cell or workspace editor. Rerun the affected route/control checks and the preservation checks. Then export a new Campus checkpoint. Do not replace the entire Campus project with a worked Live example. Your next sprint starts from the latest verified **Campus** checkpoint.

**Expected result:** the ZIP contains your Live source and safe observations, all session records remain available on resume, and your Campus workspace has not been changed by this notebook.


In [ ]:
from module_b.live_sessions import read_records
final_project = project
final_stage = 'live-limit' if final_project.name == 'live-limit' else 'auth' if final_project.name == 'auth' else 'live-project'
live_records = {**read_records(project), **read_records(final_project)}
(final_project/'fixtures/live-session-records.json').write_text(json.dumps(live_records, indent=2)+'\n')
readiness = {'structural_ready': False, 'written_evidence_present': False,
    'scope': 'live_formative_only', 'campus_assessment': 'not_evaluated',
    'session_review_status': {k:v['review_status'] for k,v in live_records.items()}}
SPRINT1_EXPORT = export_progress(final_project, REPO/'artifacts'/f'sprint-1-live', sprint=1,
    evidence={'live_sessions': live_records}, notes={'purpose':'Live workshop evidence; apply chosen changes back to Campus explicitly.'}, readiness=readiness)
print('Resume stage:', final_stage)
print('Live review status:', readiness['session_review_status'])


## Watch out for

| Observation | Check and recovery |
|---|---|
| Your new checkpoint was not imported | Existing Live work is preserved. Choose a new `RUN_NAME`, then rerun the workspace cell. |
| A later session has no prepared service | Run the named earlier guided cells or import the correct checkpoint; do not infer completion from a blank task. |
| A task still says pending | Fill your decisions and reasoning, execute its checks, then export again. |
| Runtime ended before saving | Restore your downloaded Live ZIP in a fresh runtime; saving only the notebook does not preserve workspace files. |

## Quick reference

| Action | Location |
|---|---|
| Shared code and dependency installation | `setup` |
| Import Campus or resume Live work | `workspaces` |
| Session activity and evidence | Named LS section |
| Save source and public observations | `save-checkpoint` |

## Summary

You have a separate place to rehearse, investigate and discuss FieldCare service behavior. Keep the observations tied to the source you ran, carry justified changes back into your Campus project, and preserve both your notebook and source/evidence checkpoint.
